## With structured output model

In [12]:
from pydantic import BaseModel, Field

class OutputModel(BaseModel):
    name:str = Field(description="The name of the person",max_length=20,min_length=3,default="Unknown")
    age:int = Field(..., description="Age in years",ge=7,le=45)

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # example model
    temperature=0.0,
)

llmStructured = llm.with_structured_output(OutputModel)

In [14]:
llmStructured.invoke("Tell me about a person named Ali who is 30 years old.")

OutputModel(name='Ali', age=30)

## With Output Parser

In [21]:
from langchain_core.output_parsers import PydanticOutputParser,StrOutputParser,JsonOutputParser,CommaSeparatedListOutputParser

### StrOutputParser

In [58]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
prompt = PromptTemplate(
    template="Tell me a joke on {topic} .",
    input_variables=["topic"]
)

model = ChatGroq(
    model="openai/gpt-oss-20b",  # example model    
    )

parser = StrOutputParser()

In [60]:
chain = prompt | model | parser
print(chain.invoke({"topic":"programming"}))

Why do programmers always mix up Halloween and Christmas?

Because Oct 31 == Dec 25!


### Json

In [46]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

prompt = PromptTemplate(
    template="{instructions} Tell me a joke on {topic}, and also give word count and char count .",
    input_variables=["topic","instructions"]
)

model = ChatGroq(
    model="openai/gpt-oss-20b",  # example model    
    )
parser = JsonOutputParser()

In [47]:
chain = prompt | model | parser
chain.invoke({"topic":"programming", "instructions":parser.get_format_instructions()})

{'joke': 'Why do programmers prefer dark mode? Because light attracts bugs.',
 'word_count': 10,
 'char_count': 63}

### commma Seprated

In [55]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

prompt = PromptTemplate(
    template= "{instructions} for the given topic {topic} \
     , provide a comma separated list of items related to the topic.and only proide {count} item"
)

llm = ChatGroq(
    model="openai/gpt-oss-20b",  # example model)
)

parser = CommaSeparatedListOutputParser()

chain = prompt | llm | parser
chain.invoke({"topic":"programming","count":10, "instructions":parser.get_format_instructions()})


['Python',
 'Java',
 'C++',
 'JavaScript',
 'Ruby',
 'Go',
 'Rust',
 'Swift',
 'Kotlin',
 'TypeScript']